# S3 STARE-PODS Demo — AWS S3 + RDS Postgres

Cloud counterpart of `local_starepods_examples.ipynb`. Runs the same six-step flow against real **AWS S3** (Parquet partitions) and a real **RDS Postgres** `PodsMetadata` table.

**Workflow**
1. Ingest a GMI granule → Parquet partitions on S3 + RDS metadata (with optional `clean_before_run`)
2. Find intersecting data for a bounding box via STARE SIDs + RDS
3. Download intersecting Parquet partitions from S3
4. Reconstitute an HDF5 file (both S1 and S2 scans)
5. Compare the reconstituted structure with the original granule
6. Verify RDS metadata

**Requires** `starepandas/.config` (next to this notebook) with AWS + RDS credentials, and the granule file at `GRANULE_FILE` available locally.

In [1]:
#import subprocess, sys
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
#                       "/Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_demo_and_construct_parallel",
#                       "-q"])

In [2]:
import os
import time
import h5py
from starepandas.demo_lib import StarePodsDemo
from starepandas.staredataframe import _ensure_rds_db_and_table

## Configuration

Edit these paths and parameters before running.

In [3]:
# AWS + RDS credentials. Resolves relative to this notebook's directory.
CONFIG_PATH = os.path.join(os.getcwd(), ".config")

GRANULE_FILE = (
    "/Users/thatdaihaiton/Workspace/STARE/L1C_Data_Samples/GPM/2025/Jan_1_2/"
    "1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5"
)

# S3 root where Parquet partitions and RDS metadata for this demo live.
S3_PREFIX = "s3://zarrpods/gmi-demo-parquet"

# STARE partition level used for both ingestion and bbox → SIDs lookup.
# Capped at MAX_PARTITION_LEVEL = 4 (~256 cells/granule), the regime
# where each Parquet partition is multi-MB — ideal for S3.
STARE_LEVEL = 10

# Bounding box filter — set to None to reconstitute the full granule,
# or e.g. (115, -30, 120, -25) to restrict to SW Australia / Perth.
BBOX = (115, -30, 120, -25)

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/gmi_s3_reconstituted.h5"

# Set to True to wipe S3_PREFIX (S3 objects + RDS metadata rows) before
# ingesting. Mirrors the local demo's CLEAN_BEFORE_RUN flag — prevents
# duplicate RDS rows on re-runs. Keep True unless you intentionally
# want to append more granules under the same prefix.
CLEAN_BEFORE_RUN = True

print(f"Granule  : {os.path.basename(GRANULE_FILE)}")
print(f"Datasets : {DATASETS}")
print(f"BBox     : {BBOX}  (None = full granule)")
print(f"S3 root  : {S3_PREFIX}")
print(f"Clean    : {CLEAN_BEFORE_RUN}")

Granule  : 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5
Datasets : ['GMI_S1', 'GMI_S2']
BBox     : (115, -30, 120, -25)  (None = full granule)
S3 root  : s3://zarrpods/gmi-demo-parquet
Clean    : True


## Step 1 — Ingest granule → S3 Parquet + RDS

In [4]:
%%time
demo = StarePodsDemo(aws_config_path=CONFIG_PATH)

s3_paths = demo.ingest_granules(
    data_path=GRANULE_FILE,
    instrument="GMI",
    s3_prefix=S3_PREFIX,
    level=STARE_LEVEL,
    clean_before_run=CLEAN_BEFORE_RUN,
)
print(f"Stored {len(s3_paths)} dataset path(s):")
for p in s3_paths:
    print(f"  {p}")

# Granule-specific S3 prefix so subsequent steps scope to just this granule
# (and not older ingestions that may share the same S3_PREFIX root).
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]
granule_s3_prefix = f"{S3_PREFIX}/{granule_basename}"

INFO:starepandas.demo_lib:clean_before_run=True → wiping s3://zarrpods/gmi-demo-parquet on S3 + RDS first


INFO:starepandas.demo_lib:clean_s3_prefix(s3://zarrpods/gmi-demo-parquet): deleted 514 RDS row(s), 514 S3 object(s)


INFO:starepandas.demo_lib:Ingesting GMI granules from /Users/thatdaihaiton/Workspace/STARE/L1C_Data_Samples/GPM/2025/Jan_1_2/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5


INFO:starepandas.demo_lib:Found 1 GMI files


INFO:starepandas.demo_lib:Processing 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5


...

  Progress: 200/251 partitions written...


  Progress: 250/251 partitions written...


INFO:starepandas.demo_lib:✓ Stored 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5 to ['s3://zarrpods/gmi-demo-parquet/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B', 's3://zarrpods/gmi-demo-parquet/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B']


INFO:starepandas.demo_lib:Ingested 2 Parquet dataset(s)


✓ Inserted 251 metadata rows into RDS
✓ Finished writing 251 Parquet partitions to s3://zarrpods/gmi-demo-parquet/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B
Stored 2 dataset path(s):
  s3://zarrpods/gmi-demo-parquet/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B
  s3://zarrpods/gmi-demo-parquet/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B
CPU times: user 10.6 s, sys: 676 ms, total: 11.3 s
Wall time: 1min 37s


## Step 2 — Find intersecting data via STARE SIDs

In [5]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=STARE_LEVEL)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
    intersecting = demo.find_intersecting_data(location_sids, instruments=["GMI"])
    # Scope to our granule's S3 prefix so duplicate prior runs (e.g. legacy
    # data under another prefix) don't pollute the result.
    if not intersecting.empty and "group_path" in intersecting.columns:
        intersecting = intersecting[
            intersecting["group_path"].str.startswith(granule_s3_prefix)
        ]
    print(f"Found {len(intersecting)} intersecting metadata row(s).")
else:
    location_sids = None
    intersecting = None
    print("BBOX is None — Step 4 will reconstitute the full granule directly.")

if intersecting is not None and not intersecting.empty:
    intersecting[["Dataset", "grouped_id", "group_path"]].head(8)

INFO:starepandas.demo_lib:Finding intersections for 811 SIDs across ['GMI']


INFO:starepandas.demo_lib:Coerced 811 query SIDs to 6 partition-level (level 4) grouped IDs


INFO:starepandas.demo_lib:Searching GMI metadata...


Generated 811 SIDs for bbox (115, -30, 120, -25)


INFO:starepandas.demo_lib:Found 24 intersecting partitions for GMI


INFO:starepandas.demo_lib:Found 24 total intersecting partitions


Found 12 intersecting metadata row(s).


## Step 3 — Download intersecting Parquet partitions from S3

In [6]:
%%time
if intersecting is not None and not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting["Dataset"].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting partitions to download — Step 4 will read S3 directly.")
    data_dict = {}

INFO:starepandas.demo_lib:Downloading intersecting partitions for ['GMI_S2', 'GMI_S1']


INFO:starepandas.demo_lib:Processing 6 partitions for GMI_S2


INFO:starepandas.demo_lib:✓ Combined 15356 rows for GMI_S2


INFO:starepandas.demo_lib:Processing 6 partitions for GMI_S1


INFO:starepandas.demo_lib:✓ Combined 14604 rows for GMI_S1


...

GMI_S2: 15356 rows, columns: ['lat', 'lon', 'sids', 'timestamp', 'Tc1', 'Tc2'] …


,lat,lon,sids,timestamp,Tc1,Tc2,Tc3,Tc4,Quality,incidenceAngle,...,sunLocalTime,incidenceAngleIndex1,incidenceAngleIndex2,incidenceAngleIndex3,incidenceAngleIndex4,SCstatus_SCorientation,SCstatus_SClatitude,SCstatus_SClongitude,SCstatus_SCaltitude,SCstatus_FractionalGranuleNumber
0,-34.437069,116.543686,153720859722164747,2025-01-01 03:58:00.636,288.070007,286.760010,264.600006,275.869995,0,49.41,...,11.678015,1,1,1,1,180,-31.52323,113.021034,441.016205,61567.152606
1,-34.400375,116.585487,153720531311223691,2025-01-01 03:58:00.636,288.429993,288.049988,265.690002,276.970001,0,49.41,...,11.680802,1,1,1,1,180,-31.52323,113.021034,441.016205,61567.152606
2,-34.363251,116.626717,153896903552119723,2025-01-01 03:58:00.636,288.220001,288.839996,265.029999,277.170013,0,49.41,...,11.683552,1,1,1,1,180,-31.52323,113.021034,441.016205,61567.152606


GMI_S1: 14604 rows, columns: ['lat', 'lon', 'sids', 'timestamp', 'Tc1', 'Tc2'] …


,lat,lon,sids,timestamp,Tc1,Tc2,Tc3,Tc4,Tc5,Tc6,...,incidenceAngleIndex5,incidenceAngleIndex6,incidenceAngleIndex7,incidenceAngleIndex8,incidenceAngleIndex9,SCstatus_SCorientation,SCstatus_SClatitude,SCstatus_SClongitude,SCstatus_SCaltitude,SCstatus_FractionalGranuleNumber
0,-33.991417,117.436699,153843806408674507,2025-01-01 03:58:15.636,300.079987,286.299988,301.700012,292.170013,299.070007,300.850006,...,1,1,1,1,1,180,-30.681971,113.51255,440.746246,61567.155288
1,-33.950184,117.483765,153844690609941963,2025-01-01 03:58:15.636,299.200012,285.410004,301.320007,292.649994,301.350006,301.239990,...,1,1,1,1,1,180,-30.681971,113.51255,440.746246,61567.155288
2,-33.908463,117.530190,153851494793746923,2025-01-01 03:58:15.636,299.510010,285.440002,302.130005,291.309998,301.799988,301.559998,...,1,1,1,1,1,180,-30.681971,113.51255,440.746246,61567.155288


CPU times: user 267 ms, sys: 48.2 ms, total: 315 ms
Wall time: 5.7 s


## Step 4 — Reconstitute HDF5 (S1 + S2)

In [7]:
%%time
recon_path = demo.reconstitute_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    s3_prefix=granule_s3_prefix,
)
print(f"Written to: {recon_path}")

INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S1' over bbox=(115, -30, 120, -25)


/Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_demo_and_construct_parallel/starepandas/io/granules/__init__.py:1246: UserWarning: len(df)=14604 is not divisible by pixel_width=221. Trailing 18 row(s) will be truncated.
  sdf.to_hdf5(
INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S2' over bbox=(115, -30, 120, -25)


/Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_demo_and_construct_parallel/starepandas/io/granules/__init__.py:1246: UserWarning: len(df)=15356 is not divisible by pixel_width=221. Trailing 107 row(s) will be truncated.
  sdf.to_hdf5(
INFO:starepandas.demo_lib:✓ Reconstituted HDF5 written to /tmp/gmi_s3_reconstituted.h5


Written to: /tmp/gmi_s3_reconstituted.h5
CPU times: user 949 ms, sys: 150 ms, total: 1.1 s
Wall time: 14.4 s


## Step 5 — Structure comparison: reconstituted vs original

In [8]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTITUTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")


--- RECONSTITUTED  (gmi_s3_reconstituted.h5) ---
  /S1                                                 Group
  /S1/Latitude                                        (66, 221)            float32
  /S1/Longitude                                       (66, 221)            float32
  /S1/Quality                                         (66, 221)            int8
  /S1/SCstatus                                        Group
  /S1/SCstatus/FractionalGranuleNumber                (66,)                float64
  /S1/SCstatus/SCaltitude                             (66,)                float32
  /S1/SCstatus/SClatitude                             (66,)                float32
  /S1/SCstatus/SClongitude                            (66,)                float32
  /S1/SCstatus/SCorientation                          (66,)                int16
  /S1/ScanTime                                        Group
  /S1/ScanTime/DayOfMonth                             (66,)                int8
  /S1/ScanTime/DayOfYear       

## Step 6 — RDS metadata verification

In [9]:
conn = _ensure_rds_db_and_table("StarePodsMetadata")
try:
    with conn.cursor() as cur:
        cur.execute(
            'SELECT "Dataset", COUNT(*) '
            'FROM "PodsMetadata" '
            'WHERE "MetadataJson"->>%s LIKE %s '
            'GROUP BY "Dataset" ORDER BY "Dataset"',
            ("group_path", f"{granule_s3_prefix}/%"),
        )
        rows = cur.fetchall()
    print(f"RDS prefix: {granule_s3_prefix}")
    for ds, cnt in rows:
        print(f"  {ds}: {cnt} partition(s)")
finally:
    conn.close()

RDS prefix: s3://zarrpods/gmi-demo-parquet/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B
  GMI_S1: 263 partition(s)
  GMI_S2: 251 partition(s)
